<a href="https://colab.research.google.com/github/BF667-IDLE/vsep/blob/main/notebooks/vsep_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# vsep - Audio Stem Separator Demo

Welcome to the **vsep** interactive demo! This notebook lets you separate audio into different stems (vocals, drums, bass, instruments) using state-of-the-art AI models from UVR.

## Features
- **Fast Processing** - Optimized parallel downloads and GPU-accelerated inference
- **Multiple Models** - Choose from 100+ UVR models across 4 architectures
- **Easy to Use** - Simple interface for audio separation
- **Download Results** - Get your separated stems as WAV files

See the [full documentation](https://github.com/BF667-IDLE/vsep/tree/main/docs) for more details.

---

## 1. Setup

First, let's install vsep and its dependencies. This takes about 2-3 minutes.

In [ ]:
#@title Install vsep
#@markdown Run this cell to install vsep (first time only)

# Clone the repository
!git clone -q https://github.com/BF667-IDLE/vsep.git /content/vsep

# Install dependencies
!pip install -q -r /content/vsep/requirements.txt

# Install ffmpeg (required by pydub for audio I/O)
!apt-get -qq install ffmpeg > /dev/null 2>&1

import os
os.chdir('/content/vsep')

import torch
if torch.cuda.is_available():
    print(f"NVIDIA GPU detected: {torch.cuda.get_device_name(0)}")
else:
    print("No GPU detected, will use CPU (slower)")
    print("Tip: Go to Runtime > Change runtime type > Hardware accelerator > GPU")

## 2. Upload Your Audio

Upload the audio file you want to separate. Supported formats: MP3, WAV, FLAC, OGG, M4A

In [ ]:
#@title Upload Audio File
#@markdown Upload your audio file for separation

from google.colab import files
import os

print("Please upload your audio file...")
uploaded = files.upload()

if uploaded:
    audio_file = list(uploaded.keys())[0]
    print(f"\nUploaded: {audio_file}")
    os.makedirs("output", exist_ok=True)
else:
    print("No file uploaded. Please run this cell again.")

## 3. Choose Model

Select the model you want to use for separation. Different models work better for different types of audio.

In [ ]:
#@title Select Model
#@markdown Choose a separation model

model_choice = "model_bs_roformer_ep_317_sdr_12.9755.ckpt" #@param ["model_bs_roformer_ep_317_sdr_12.9755.ckpt", "Mel-Roformer-Viperx-1053.ckpt", "ht-demucs_ft.yaml", "UVR-MDX-NET-Inst_1.onnx", "BS-Roformer-Viperx-1297.ckpt", "Kim_Vocal_2.onnx"]

model_descriptions = {
    "model_bs_roformer_ep_317_sdr_12.9755.ckpt": "BS-Roformer - Best overall vocal quality (SDR 12.98, 2 stems)",
    "Mel-Roformer-Viperx-1053.ckpt": "Mel-Roformer - High-quality vocal separation (SDR 12.61, 2 stems)",
    "ht-demucs_ft.yaml": "Demucs v4 - Best all-rounder (4 stems: vocals, drums, bass, other)",
    "UVR-MDX-NET-Inst_1.onnx": "MDX-Net - Good instrumental/vocal split (SDR 10.65, 2 stems)",
    "BS-Roformer-Viperx-1297.ckpt": "BS-Roformer Viperx - Excellent vocal isolation (2 stems)",
    "Kim_Vocal_2.onnx": "Kim Vocal - Specialized for vocal extraction (2 stems)"
}

print(f"Selected Model: {model_choice}")
print(f"Description: {model_descriptions.get(model_choice, 'Custom model')}")
selected_model = model_choice

## 4. Separate Audio

Now let's separate your audio file using the selected model.

In [ ]:
#@title Run Separation
#@markdown Click to start the separation process

from separator import Separator
import config.variables as cfg

# Optimize for Colab
cfg.MAX_DOWNLOAD_WORKERS = 4
cfg.DOWNLOAD_CHUNK_SIZE = 262144

print("Starting separation process...")
print(f"Input file: {audio_file}")
print(f"Model: {selected_model}")
print("\nThis may take 2-5 minutes depending on the model and audio length...\n")

# Initialize separator
separator = Separator(
    model_file_dir="/content/vsep/models",
    output_dir="/content/vsep/output",
    output_format="WAV",
    use_soundfile=True,
)

# Load model
separator.load_model(model_filename=selected_model)

# Run separation
output_files = separator.separate(audio_file)

print(f"\nSeparation complete!")
print(f"Output files: {len(output_files)}")
for f in output_files:
    print(f"  - {f}")

## 5. Listen to Results

Play back the separated stems to hear the results.

In [ ]:
#@title Listen to Separated Stems
#@markdown Browse and play the separated audio files

import glob
from IPython.display import Audio, display

# Find all output files (WAV, FLAC, MP3)
output_files = sorted(glob.glob("/content/vsep/output/*.*"))

if output_files:
    print(f"Found {len(output_files)} separated stems:\n")
    for i, file_path in enumerate(output_files, 1):
        filename = os.path.basename(file_path)
        print(f"{i}. {filename}")
        display(Audio(file_path))
        print("-" * 50)
else:
    print("No output files found. Please run the separation first.")

## 6. Download Results

Download the separated stems to your computer.

In [ ]:
#@title Download Separated Stems
#@markdown Download all separated audio files

from google.colab import files
import glob

output_files = glob.glob("/content/vsep/output/*.*")

if output_files:
    print(f"Downloading {len(output_files)} files...")
    for file_path in output_files:
        files.download(file_path)
        print(f"Downloaded: {os.path.basename(file_path)}")
else:
    print("No files to download. Please run separation first.")

In [ ]:
#@title List All Available Models (Optional)
#@markdown Run this cell to see all 100+ supported models

from separator import Separator

sep = Separator(info_only=True)
models = sep.get_simplified_model_list(filter_sort_by="vocals")

print(f"Showing top 20 vocal models (out of {len(models)} total):\n")
print(f"{'Model Filename':<55} {'Architecture':<10} {'Stems'}")
print("-" * 100)
for i, (filename, info) in enumerate(list(models.items())[:20], 1):
    stems = ", ".join(info["Stems"])
    print(f"{filename:<55} {info['Type']:<10} {stems}")

## Additional Information

### Model Recommendations

| Use Case | Recommended Model | Stems |
|:---------|:------------------|:------|
| Best vocals | `model_bs_roformer_ep_317_sdr_12.9755.ckpt` | vocals, instrumental |
| All instruments | `ht-demucs_ft.yaml` | vocals, drums, bass, other |
| Karaoke track | `UVR-MDX-NET-Inst_1.onnx` | vocals, instrumental |
| Complex mixes | `Mel-Roformer-Viperx-1053.ckpt` | vocals, instrumental |

### Tips

1. **GPU Acceleration:** Make sure you're using a GPU runtime for faster processing
   - Go to `Runtime > Change runtime type > Hardware accelerator > GPU`
2. **Long Files:** For files longer than 10 minutes, enable chunking to avoid memory errors
3. **Quality vs Speed:** Larger Roformer models produce better quality but take longer
4. **Memory:** If you run out of memory, try a smaller model (MDX-Net) or shorter audio

### Troubleshooting

**"No GPU detected"**
- Go to Runtime > Change runtime type > GPU
- Restart the runtime if needed

**"Download failed"**
- The model download may have timed out - re-run the separation cell (downloads are cached)
- Increase timeout: `cfg.DOWNLOAD_TIMEOUT = 600`

**"Out of memory"**
- Try a shorter audio file or use `chunk_duration=300` when creating the Separator
- Use a simpler model (MDX-Net instead of Roformer)

**"Module not found"**
- Make sure you ran the installation cell first
- Try restarting the runtime (Runtime > Restart runtime)

---

### Links

- **GitHub:** [github.com/BF667-IDLE/vsep](https://github.com/BF667-IDLE/vsep)
- **API Reference:** [docs/API-Reference.md](https://github.com/BF667-IDLE/vsep/blob/main/docs/API-Reference.md)
- **Architecture:** [docs/Architecture.md](https://github.com/BF667-IDLE/vsep/blob/main/docs/Architecture.md)
- **Install Guide:** [INSTALL.md](https://github.com/BF667-IDLE/vsep/blob/main/INSTALL.md)
- **Issues:** [github.com/BF667-IDLE/vsep/issues](https://github.com/BF667-IDLE/vsep/issues)

Made with care using vsep